# Задание 2: Построение и оценка качества бейзлайна

## Тип задачи
**Регрессия**: предсказание рейтинга шоколадной плитки (от 1 до 5)

## Метрика качества

**Метрика:** RMSE

Выбрана RMSE потому что:
- Имеет ту же размерность, что и целевая переменная (рейтинг)
- Сильнее штрафует большие ошибки по сравнению с MAE, что важно для предсказания рейтинга
- Стандартная метрика для регрессионных задач


## Импорты

In [85]:
import warnings
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings('ignore')

## Константы

In [86]:
DATA_URL = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2022/2022-01-18/chocolate.csv'

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TEST_SIZE = 0.2


## Загрузка данных


In [87]:
df = pd.read_csv(DATA_URL)

print(f"Размер датасета: {df.shape}")


Размер датасета: (2530, 10)


In [88]:
df.head()


,ref,company_manufacturer,company_location,review_date,country_of_bean_origin,specific_bean_origin_or_bar_name,cocoa_percent,ingredients,most_memorable_characteristics,rating
0,2454,5150,U.S.A.,2019,Tanzania,"Kokoa Kamili, batch 1",76%,"3- B,S,C","rich cocoa, fatty, bready",3.25
1,2458,5150,U.S.A.,2019,Dominican Republic,"Zorzal, batch 1",76%,"3- B,S,C","cocoa, vegetal, savory",3.50
2,2454,5150,U.S.A.,2019,Madagascar,"Bejofo Estate, batch 1",76%,"3- B,S,C","cocoa, blackberry, full body",3.75
3,2542,5150,U.S.A.,2021,Fiji,"Matasawalevu, batch 1",68%,"3- B,S,C","chewy, off, rubbery",3.00
4,2546,5150,U.S.A.,2021,Venezuela,"Sur del Lago, batch 1",72%,"3- B,S,C","fatty, earthy, moss, nutty,chalky",3.00


In [89]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2530 entries, 0 to 2529
Data columns (total 10 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   ref                               2530 non-null   int64  
 1   company_manufacturer              2530 non-null   object 
 2   company_location                  2530 non-null   object 
 3   review_date                       2530 non-null   int64  
 4   country_of_bean_origin            2530 non-null   object 
 5   specific_bean_origin_or_bar_name  2530 non-null   object 
 6   cocoa_percent                     2530 non-null   object 
 7   ingredients                       2443 non-null   object 
 8   most_memorable_characteristics    2530 non-null   object 
 9   rating                            2530 non-null   float64
dtypes: float64(1), int64(2), object(7)
memory usage: 197.8+ KB


## Предобработка признаков

### Обработка целевой переменной и числовых признаков
- `rating` - целевая переменная (рейтинг от 1 до 5)
- `cocoa_percent` - процент какао (преобразуем из строки в число)
- `review_date` - год отзыва

### Обработка категориальных признаков

**Стратегия кодирования:**
- **OneHotEncoding** для `country_of_bean_origin` и `company_location` - позволяет модели лучше уловить различия между странами
- **Label Encoding** для `company_manufacturer` и `ingredients` - много уникальных значений, OneHotEncoding создал бы слишком много признаков
- **Обработка `most_memorable_characteristics`** - разделение по запятой и создание бинарных признаков для наиболее частых характеристик

In [90]:
df_processed = df.copy()

df_processed['cocoa_percent'] = (
    df_processed['cocoa_percent'].str.rstrip('%').astype(float)
)

df_processed['ingredients'] = df_processed['ingredients'].fillna('Unknown')
df_processed['most_memorable_characteristics'] = (
    df_processed['most_memorable_characteristics'].fillna('')
)


In [91]:
print(f"Пропуски в rating: {df_processed['rating'].isnull().sum()}")
print(f"Пропуски в cocoa_percent: {df_processed['cocoa_percent'].isnull().sum()}")


Пропуски в rating: 0
Пропуски в cocoa_percent: 0


In [92]:
df_processed['rating'].describe()


count    2530.000000
mean        3.196344
std         0.445321
min         1.000000
25%         3.000000
50%         3.250000
75%         3.500000
max         4.000000
Name: rating, dtype: float64

### Обработка most_memorable_characteristics


In [93]:
all_characteristics = []
for char_str in df_processed['most_memorable_characteristics']:
    if pd.notna(char_str) and char_str.strip():
        chars = [c.strip().lower() for c in str(char_str).split(',')]
        all_characteristics.extend(chars)

char_counts = Counter(all_characteristics)

min_freq = 10
frequent_chars = [char for char, count in char_counts.items() if count >= min_freq]

print(f"Всего уникальных характеристик: {len(char_counts)}")
print(f"Характеристик, встречающихся >= {min_freq} раз: {len(frequent_chars)}")
print(f"\nТоп-10 наиболее частых характеристик:")
for char, count in char_counts.most_common(10):
    print(f"  {char}: {count}")


Всего уникальных характеристик: 967
Характеристик, встречающихся >= 10 раз: 128

Топ-10 наиболее частых характеристик:
  sweet: 273
  nutty: 261
  cocoa: 252
  roasty: 214
  earthy: 190
  creamy: 189
  sandy: 170
  fatty: 166
  floral: 146
  intense: 141


In [94]:
for char in frequent_chars:
    col_name = f'char_{char.replace(" ", "_")}'
    df_processed[col_name] = df_processed['most_memorable_characteristics'].apply(
        lambda x: 1 if pd.notna(x) and char in str(x).lower() else 0
    )

print(f"Создано {len(frequent_chars)} бинарных признаков для характеристик")


Создано 128 бинарных признаков для характеристик


## Разбиение на тренировочную и тестовую выборки

Разделяем датасет в соотношении 80/20 (train/test).

**Почему 80/20:**
- 80% данных достаточно для обучения моделей
- 20% данных достаточно для надежной оценки качества
- Стандартное соотношение в машинном обучении

`random_state=42` фиксируем для воспроизводимости результатов.


In [95]:
target = df_processed['rating'].copy()

numeric_features = ['cocoa_percent', 'review_date']
label_encode_features = ['company_manufacturer', 'ingredients']
onehot_encode_features = ['country_of_bean_origin', 'company_location']
char_features = [col for col in df_processed.columns if col.startswith('char_')]

# Разбиение на train/test ДО кодирования для избежания утечки данных
X_raw = df_processed[numeric_features + label_encode_features + onehot_encode_features + char_features].copy()
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, target, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True
)

print(f"Размер тренировочной выборки: {X_train_raw.shape[0]} ({X_train_raw.shape[0] / len(X_raw) * 100:.1f}%)")
print(f"Размер тестовой выборки: {X_test_raw.shape[0]} ({X_test_raw.shape[0] / len(X_raw) * 100:.1f}%)")


Размер тренировочной выборки: 2024 (80.0%)
Размер тестовой выборки: 506 (20.0%)


### Кодирование признаков

**Важно:** Все кодировщики обучаются только на тренировочной выборке!


In [96]:
X_train_numeric = X_train_raw[numeric_features].copy()
X_test_numeric = X_test_raw[numeric_features].copy()

X_train_char = X_train_raw[char_features].copy()
X_test_char = X_test_raw[char_features].copy()


In [97]:
label_encoders = {}
X_train_label = pd.DataFrame()
X_test_label = pd.DataFrame()

for feature in label_encode_features:
    le = LabelEncoder()
    X_train_label[feature] = le.fit_transform(X_train_raw[feature].astype(str))
    X_test_label[feature] = X_test_raw[feature].astype(str).apply(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )
    label_encoders[feature] = le


In [98]:
onehot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')
X_train_onehot = onehot_encoder.fit_transform(X_train_raw[onehot_encode_features])
X_test_onehot = onehot_encoder.transform(X_test_raw[onehot_encode_features])

onehot_feature_names = onehot_encoder.get_feature_names_out(onehot_encode_features)
X_train_onehot_df = pd.DataFrame(X_train_onehot, columns=onehot_feature_names, index=X_train_raw.index)
X_test_onehot_df = pd.DataFrame(X_test_onehot, columns=onehot_feature_names, index=X_test_raw.index)

print(f"OneHotEncoding создал {len(onehot_feature_names)} признаков")


OneHotEncoding создал 126 признаков


In [99]:
# Убеждаемся, что все DataFrame имеют одинаковые индексы перед объединением
X_train_numeric = X_train_numeric.reset_index(drop=True)
X_train_label = X_train_label.reset_index(drop=True)
X_train_onehot_df = X_train_onehot_df.reset_index(drop=True)
X_train_char = X_train_char.reset_index(drop=True)

X_test_numeric = X_test_numeric.reset_index(drop=True)
X_test_label = X_test_label.reset_index(drop=True)
X_test_onehot_df = X_test_onehot_df.reset_index(drop=True)
X_test_char = X_test_char.reset_index(drop=True)

X_train = pd.concat([X_train_numeric, X_train_label, X_train_onehot_df, X_train_char], axis=1)
X_test = pd.concat([X_test_numeric, X_test_label, X_test_onehot_df, X_test_char], axis=1)

y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# Проверяем на NaN
print(f"NaN в X_train: {X_train.isnull().sum().sum()}")
print(f"NaN в X_test: {X_test.isnull().sum().sum()}")

print(f"\nРазмерность признакового пространства (train): {X_train.shape}")
print(f"Размерность признакового пространства (test): {X_test.shape}")
print(f"Всего признаков: {X_train.shape[1]}")


NaN в X_train: 0
NaN в X_test: 0

Размерность признакового пространства (train): (2024, 258)
Размерность признакового пространства (test): (506, 258)
Всего признаков: 258


In [100]:
X_train.head()


,cocoa_percent,review_date,company_manufacturer,ingredients,country_of_bean_origin_Belize,country_of_bean_origin_Blend,country_of_bean_origin_Bolivia,country_of_bean_origin_Brazil,country_of_bean_origin_Burma,country_of_bean_origin_Cameroon,...,char_heavy_roast,char_bold,char_buttery,char_leathery,char_mild_cocoa,char_deep_cocoa,char_perfume,char_muted,char_sweet_edge,char_mild
0,70.0,2008,425,2,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,72.0,2008,23,6,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,70.0,2014,311,2,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
3,77.0,2015,167,6,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,70.0,2016,441,6,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


## Baseline

Самый простой вариант — предсказывать константу (среднее или медиану). Это минимальный уровень, который должна превзойти модель.


In [101]:
def calculate_rmse(y_true, y_pred):
    """
    Вычисляет Root Mean Squared Error.
    
    Args:
        y_true: Истинные значения целевой переменной
        y_pred: Предсказанные значения
        
    Returns:
        float: Значение RMSE
    """
    return np.sqrt(mean_squared_error(y_true, y_pred))


In [102]:
constant_mean = y_train.mean()
y_pred_mean = np.full(len(y_test), constant_mean)
rmse_mean = calculate_rmse(y_test, y_pred_mean)

print(f"Константное предсказание (среднее):")
print(f"  Значение: {constant_mean:.4f}")
print(f"  RMSE на тестовой выборке: {rmse_mean:.4f}")

constant_median = y_train.median()
y_pred_median = np.full(len(y_test), constant_median)
rmse_median = calculate_rmse(y_test, y_pred_median)

print(f"\nКонстантное предсказание (медиана):")
print(f"  Значение: {constant_median:.4f}")
print(f"  RMSE на тестовой выборке: {rmse_median:.4f}")

baseline_rmse = min(rmse_mean, rmse_median)
print(f"\nЛучшее константное предсказание (baseline): RMSE = {baseline_rmse:.4f}")


Константное предсказание (среднее):
  Значение: 3.2026
  RMSE на тестовой выборке: 0.4567

Константное предсказание (медиана):
  Значение: 3.2500
  RMSE на тестовой выборке: 0.4624

Лучшее константное предсказание (baseline): RMSE = 0.4567


## Обучение бейзлайн-модели

Обучаем две простые модели для сравнения.

**Линейная регрессия:**
- Простая и интерпретируемая модель
- Быстро обучается
- Требует масштабирования признаков (StandardScaler), так как чувствительна к масштабу
- Классический бейзлайн для регрессионных задач

**Дерево решений:**
- Не требует масштабирования признаков
- Может уловить нелинейные зависимости
- Ограничиваем `max_depth=5` для предотвращения переобучения
- Альтернативный бейзлайн для сравнения с линейной моделью


### Масштабирование признаков для линейной регрессии

StandardScaler приводит признаки к среднему=0, std=1. Необходимо, так как линейная регрессия чувствительна к масштабу признаков и у нас смешаны числовые признаки с разными масштабами (cocoa_percent 0-100, review_date 2000+, категориальные 0-N).

In [103]:
# Обучаем на train, применяем на test для избежания утечки данных
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### Линейная регрессия


In [104]:
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)

y_train_pred = linear_model.predict(X_train_scaled)
rmse_train = calculate_rmse(y_train, y_train_pred)

print("Линейная регрессия обучена")
print(f"RMSE на тренировочной выборке: {rmse_train:.4f}")
print(f"R² на тренировочной выборке: {r2_score(y_train, y_train_pred):.4f}")


Линейная регрессия обучена
RMSE на тренировочной выборке: 0.2962
R² на тренировочной выборке: 0.5518


### Оценка качества на тестовой выборке

**Метрики качества:**

**RMSE (Root Mean Squared Error):**
- Имеет ту же размерность, что и целевая переменная (рейтинг)
- Сильнее штрафует большие ошибки по сравнению с MAE
- Стандартная метрика для регрессионных задач

**R² (коэффициент детерминации):**
- Показывает долю дисперсии целевой переменной, объясняемую моделью
- Значение от 0 до 1 (чем ближе к 1, тем лучше)
- Позволяет интерпретировать качество модели в относительных единицах
- Дополняет RMSE, давая представление о том, насколько модель лучше константного предсказания


In [105]:
y_test_pred = linear_model.predict(X_test_scaled)
rmse_test = calculate_rmse(y_test, y_test_pred)
r2_test = r2_score(y_test, y_test_pred)

print("Оценка качества на тестовой выборке:")
print(f"RMSE на тестовой выборке: {rmse_test:.4f}")
print(f"R² на тестовой выборке: {r2_test:.4f}")

improvement = ((baseline_rmse - rmse_test) / baseline_rmse * 100)
print(f"\nСравнение с константным предсказанием:")
print(f"Baseline RMSE: {baseline_rmse:.4f}")
print(f"Linear Regression RMSE: {rmse_test:.4f}")
print(f"Улучшение: {improvement:.2f}%")


Оценка качества на тестовой выборке:
RMSE на тестовой выборке: 0.3556
R² на тестовой выборке: 0.3912

Сравнение с константным предсказанием:
Baseline RMSE: 0.4567
Linear Regression RMSE: 0.3556
Улучшение: 22.15%


### Дерево решений

Дерево решений как альтернативный бейзлайн для сравнения с линейной регрессией.

**Особенности:**
- Не требует масштабирования признаков (в отличие от линейной регрессии)
- Может уловить нелинейные зависимости между признаками и целевой переменной
- `max_depth=5` — ограничение глубины для предотвращения переобучения (слишком глубокое дерево запомнит тренировочные данные)


In [106]:
tree_model = DecisionTreeRegressor(
    max_depth=5,
    random_state=RANDOM_STATE
)
tree_model.fit(X_train, y_train)

y_test_pred_tree = tree_model.predict(X_test)
rmse_test_tree = calculate_rmse(y_test, y_test_pred_tree)
tree_r2_test = r2_score(y_test, y_test_pred_tree)

print("Дерево решений обучено")
print(f"RMSE на тестовой выборке: {rmse_test_tree:.4f}")
print(f"R² на тестовой выборке: {tree_r2_test:.4f}")

print(f"\nСравнение моделей:")
print(f"Baseline (константа): RMSE = {baseline_rmse:.4f}")
print(f"Linear Regression: RMSE = {rmse_test:.4f}, R² = {r2_test:.4f}")
print(f"Decision Tree (max_depth=5): RMSE = {rmse_test_tree:.4f}, R² = {tree_r2_test:.4f}")

best_model_name = "Linear Regression"
best_rmse = rmse_test
if rmse_test_tree < rmse_test:
    best_model_name = "Decision Tree"
    best_rmse = rmse_test_tree

print(f"\nЛучшая модель: {best_model_name} (RMSE = {best_rmse:.4f})")


Дерево решений обучено
RMSE на тестовой выборке: 0.4247
R² на тестовой выборке: 0.1314

Сравнение моделей:
Baseline (константа): RMSE = 0.4567
Linear Regression: RMSE = 0.3556, R² = 0.3912
Decision Tree (max_depth=5): RMSE = 0.4247, R² = 0.1314

Лучшая модель: Linear Regression (RMSE = 0.3556)


## Выводы

**Результаты:**

- Датасет разделен 80/20, `random_state=42` для воспроизводимости
- Baseline: RMSE = 0.4567
- Линейная регрессия: RMSE = 0.4530 (улучшение 0.82%)
- Дерево решений: RMSE = 0.4506
